In [ ]:
# === Cell 1: Setup & selections ==============================================
# Choose your scenario FOLDERS (first must be the baseline)
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# -----------------------------------------------------------------------------
# User inputs General for all results
# -----------------------------------------------------------------------------

# === Single source of truth: scenarios with path + label ===
# First entry is always the baseline
scenarios = [
    # { "dir": r"baselines/IEC_IC_baseline_IEA22MW","label": "Baseline IEC",},   
    # { "dir": r"results/HKN_codesign_3147_WSWD_TI_dist_3deg_TIboost", "label": "Case 3147",},
    # { "dir": r"results/HKN_codesign_6725_WSWD_TI_dist_3deg_TIboost", "label": "Case 6725",},
    # { "dir": r"results/HKN_codesign_8649_WSWD_TI_dist_3deg_TIboost", "label": "Case 8649",},
    # { "dir": r"results/HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost", "label": "Case 6290",},


#    { "dir": r"baselines/TS_FA_HKN_v4_TIboost" ,"label": "Baseline operation",}, 
#    { "dir":  r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_DA_yaw_lookup_Pmax_fixedTI_v4_TIboost","label":"Wake steerin operation" ,}, 

    #{ "dir": r"baselines/TS_FA_HKN_v4_TIboost" ,"label": "Baseline operation",}, 
    { "dir": r"baselines/TS_DA_HKN_full_filled_small_gaps_only" ,"label": "Baseline operation",}, 
    { "dir": r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_TS_DA_HKN_Revenue_shutdown_500_TI_boost" ,"label": "Threshold 500 eur" ,},
    { "dir": r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_TS_DA_HKN_Revenue_shutdown_800_TI_boost" ,"label": "Threshold 800 eur",},
    { "dir": r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_TS_DA_HKN_Revenue_shutdown_1000_TI_boost","label": "Threshold 1000 eur",},
    { "dir": r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_TS_DA_HKN_Revenue_shutdown_1200_TI_boost","label": "Threshold 1200 eur",},
    { "dir": r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_TS_DA_HKN_Revenue_shutdown_1400_TI_boost","label": "Threshold 1400 eur",},
    { "dir": r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_TS_DA_HKN_Revenue_shutdown_1600_TI_boost","label": "Threshold 1600 eur",},



    # { "dir": ,"label": ,},
]

scenario_dirs   = [s["dir"]   for s in scenarios]   # for loading data
scenario_labels = [s["label"] for s in scenarios]   # for legend/text


# All available channels (internal names)
all_channels = [
    "RA_ADC", "RA_BRM", "RA_ebrm", "RA_fbrm", "RA_blade_torsion", "RA_tbfa",
    "RA_tbss", "RA_TBM", "RA_TB_torsion", "RA_ttfa", "RA_TTM", "RA_gsb_l10",
    "RA_rsb_l10", "RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed", "Energy",
    "Revenue", "YawTravel"
]

# Select the channels to plot (subset of all_channels)
selected_channels = [
    # "RA_tbfa", 
    # "RA_tbss", 
    "RA_TBM", 
    # "RA_TB_torsion",
    # "RA_ttfa", 
    "RA_TTM", 
    # "RA_ebrm", 
    # "RA_fbrm", 
    "RA_BRM",
    "RA_blade_torsion", 
    "RA_gsb_l10",
    "RA_rsb_l10", 
    "RA_shaft_mx_mb_fixed", 
    "RA_shaft_mz_mb_fixed", 
    "RA_ADC", 
    "YawTravel",
    "Energy", 
    "Revenue",
]



# Map internal channel names -> display names (fill in as you like)
# If a channel is missing here, its internal name will be used by default.
display_name = {
    # Example custom labels:
    "RA_tbfa": "TB fore-aft",
    "RA_tbss": "TB side-side",
    "RA_TBM": "TB proj",
    "RA_TB_torsion": "TB torsion",
    "RA_ttfa": "TT fore-aft",
    "RA_TTM": "TT proj",
    "RA_ebrm": "BR edge",
    "RA_fbrm": "BR flap",
    "RA_BRM": "BR proj",
    "RA_blade_torsion": "BR torsion",
    "RA_shaft_mx_mb_fixed": "Shaft lat mom",
    "RA_shaft_mz_mb_fixed": "Shaft torsion",
    "RA_gsb_l10": "GS Bearing L10",
    "RA_rsb_l10": "RS Bearing L10",
    "RA_ADC": "Pitch ADC",
    "Energy":  "Energy",
    "Revenue": "Revenue",
    "YawTravel": "Yaw Travel"
}

# Channels that should always appear at the end (right side) if present
tail_priority = ["PitchADC", "YawTravel", "Energy", "Revenue"]

# Filenames expected inside each scenario folder
FATIGUE_NAME = "fatigue_budget.json"
TURBINE_METRICS_NAME = "turbine_metrics.parquet"

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def ordered_channels(chs, tail=tail_priority):
    """Return channels ordered so that any in `tail` appear at the end, preserving input order."""
    head = [c for c in chs if c not in tail]
    tail_kept = [c for c in tail if c in chs]
    return head + tail_kept

def labelize(chs):
    """Map internal channel names to display labels using `display_name` dict."""
    return [display_name.get(c, c) for c in chs]

# def scenario_labels_from_dirs(dirs):
#     """Human-friendly labels from folder names."""
#     return [Path(d).name for d in dirs]

# scenario_labels = scenario_labels_from_dirs(scenario_dirs)
channels_ordered = ordered_channels([c for c in selected_channels if c in all_channels])
labels_ordered = labelize(channels_ordered)

print("Scenarios (baseline first):", scenario_labels)
print("Channels (ordered):", channels_ordered)
print("Display labels:", labels_ordered)


Scenarios (baseline first): ['Baseline operation', 'Threshold 500 eur', 'Threshold 800 eur', 'Threshold 1000 eur', 'Threshold 1200 eur', 'Threshold 1400 eur', 'Threshold 1600 eur']
Channels (ordered): ['RA_TBM', 'RA_TTM', 'RA_BRM', 'RA_blade_torsion', 'RA_gsb_l10', 'RA_rsb_l10', 'RA_shaft_mx_mb_fixed', 'RA_shaft_mz_mb_fixed', 'RA_ADC', 'YawTravel', 'Energy', 'Revenue']
Display labels: ['TB proj', 'TT proj', 'BR proj', 'BR torsion', 'GS Bearing L10', 'RS Bearing L10', 'Shaft lat mom', 'Shaft torsion', 'Pitch ADC', 'Yaw Travel', 'Energy', 'Revenue']


In [14]:
# === Cell 2: Fatigue budget relative differences =============================
# Visual controls
FIG_WIDTH = 1200
FIG_HEIGHT = 600
FONT_SIZE = 18
MARKER_SIZE = 10
MARKER_SYMBOL = "triangle-up"
LEGEND_X = 1.02          # move legend to the right of plot area
LEGEND_Y = 1.0
LEGEND_ORIENTATION = "v"  # "v" or "h"
ZERO_LINE_WIDTH = 1       # thick black zero line
SHOW_LEGEND = True
SHOW_TITLE = True  # or False

# ----------------------------------------------------------------------------- 
# Load fatigue_budget.json from each scenario folder and compute rel diffs
# -----------------------------------------------------------------------------
fatigue_files = [str(Path(d) / FATIGUE_NAME) for d in scenario_dirs]

# Load as DataFrame of dicts
fatigue_rows = []
for f in fatigue_files:
    with open(f, "r") as fh:
        fatigue_rows.append(json.load(fh))
fatigue_df = pd.DataFrame(fatigue_rows)

# Keep only the chosen channels (intersect with what exists)
available = [c for c in channels_ordered if c in fatigue_df.columns]
if not available:
    raise ValueError("None of the selected channels exist in fatigue_budget.json.")
fatigue_df = fatigue_df[available]

# Baseline row
base = fatigue_df.iloc[0]

# Relative differences for scenarios (rows 1..)
rel = 100.0 * (fatigue_df.iloc[1:, :].subtract(base, axis=1)) / base.replace(0, np.nan)
rel.index = scenario_labels[1:]  # nice index

# For plotting, rename columns to display labels
colmap = dict(zip(available, labelize(available)))
rel = rel.rename(columns=colmap)

# ----------------------------------------------------------------------------- 
# Build the figure
# -----------------------------------------------------------------------------
fig = go.Figure()

x_order = [colmap[c] for c in available]  # ensure ordered x
for scen_name, row in rel.iterrows():
    fig.add_trace(go.Scatter(
        x=x_order,
        y=[row[c] for c in x_order],
        mode="markers",
        marker=dict(size=MARKER_SIZE, symbol=MARKER_SYMBOL),
        name=scen_name,
        showlegend=True
    ))

# Add a thick black zero line
fig.add_hline(y=0, line_width=ZERO_LINE_WIDTH, line_color="black")

if SHOW_LEGEND:
    legend=dict(x=LEGEND_X, y=LEGEND_Y, orientation=LEGEND_ORIENTATION)
else:
    legend = dict(visible=False)

fig.update_layout(
    title="Relative Difference vs Baseline — Fatigue Budget" if SHOW_TITLE else None,
    xaxis_title="Channel",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    font=dict(size=FONT_SIZE),
    legend=legend
)

fig.update_xaxes(tickangle=-45)
fig.show()


In [15]:
# === Cell 3 grouped boxes of relative per turbine (with eachself baseline) and channel ===
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

# --- Visual controls ---------------------------------------------------------
FIG_WIDTH  = 1400
FIG_HEIGHT = 720
FONT_SIZE  = 18

# Box geometry & style
BOX_LINE_WIDTH = 1.6
BOX_ALPHA      = 0.30               # face alpha
BOX_OFFSET     = 0.24               # half-offset between scenarios within a channel
BOX_WIDTH      = 0.32               # **controls actual box width** (edge-to-edge overlays use this)

# Mean / Median overlays (only these two lines)
DRAW_MEAN        = True
DRAW_MEDIAN      = True
MEAN_LINE_WIDTH  = 2.2
MEAN_LINE_DASH   = "solid"
MEDIAN_LINE_WIDTH= 3.0
MEDIAN_LINE_DASH = "dot"

# Scatter to the RIGHT of each scenario box
SCATTER_OVERLAY   = False
SCATTER_SIZE      = 6
SCATTER_OPACITY   = 0.75
SCATTER_RIGHT_GAP = 0.1           # distance to the right of box edge

# Legend
LEGEND_X = 1.02
LEGEND_Y = 1.00
LEGEND_ORIENTATION = "v"
SHOW_LEGEND = True
SHOW_TITLE = True

# --- Color control -----------------------------------------------------------
# You can override the first up to 5 scenario colors here (hex or rgb/rgba):
SCENARIO_COLORS_USER = [

        "#7f7f7f",  # grey
        "#1f77b4",  # 2nd (blue)
        "#2ca02c",  # strong green
        "#d62728",  # red
        "#9467bd",  # 5th (purple)
        # "#66c2a5",  # soft green (greenish/teal)
        # "#1f77b4",  # blue    
    #  "#ff7f0e",  # 3rd (orange)
    # "#2ca02c",  # 1st scenario color (green-ish)
    
   
    # "#d62728",  # 4th (red)
    # "#9467bd",  # 5th (purple)
]

def make_transparent(color_str, alpha=0.3):
    """Return a transparent version of a Plotly color (hex or rgb/rgba)."""
    s = color_str.strip()
    if s.startswith("rgba"):
        parts = s[5:-1].split(",")
        r, g, b = [p.strip() for p in parts[:3]]
        return f"rgba({r},{g},{b},{alpha})"
    if s.startswith("rgb("):
        parts = s[4:-1].split(",")
        r, g, b = [p.strip() for p in parts]
        return f"rgba({r},{g},{b},{alpha})"
    # assume hex
    hex_color = s.lstrip("#")
    r = int(hex_color[0:2], 16); g = int(hex_color[2:4], 16); b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

# --- Data (reuses variables from Cell 1) -------------------------------------
# Needed from Cell 1: scenario_dirs, scenario_labels, channels_ordered, display_name
TURBINE_METRICS_NAME = "turbine_metrics.parquet"
tm_files  = [str(Path(d) / TURBINE_METRICS_NAME) for d in scenario_dirs]
tm_frames = [pd.read_parquet(p) for p in tm_files]

def detect_tid_col(df):
    for c in df.columns:
        if c.lower() in ("id", "turbine", "turbine_id", "tid"):
            return c
    return None

tid_col = detect_tid_col(tm_frames[0]) or "turbine_id"
if tid_col not in tm_frames[0].columns:
    tm_frames[0] = tm_frames[0].assign(**{tid_col: range(len(tm_frames[0]))})

channels_for_plot = [c for c in channels_ordered if c in tm_frames[0].columns]
if not channels_for_plot:
    raise ValueError("None of the selected channels exist in turbine_metrics.parquet.")
labels_for_plot = [display_name.get(c, c) for c in channels_for_plot]
label_map = dict(zip(channels_for_plot, labels_for_plot))

x_base   = np.arange(len(channels_for_plot))
tickvals = x_base
ticktext = labels_for_plot

case_labels = scenario_labels[1:]
n_cases = len(case_labels)
offsets = np.linspace(-BOX_OFFSET, BOX_OFFSET, n_cases) if n_cases > 1 else np.array([0.0])

# Build a per-scenario color map using your preferred first 5 colors
fallback_palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Dark24
palette = SCENARIO_COLORS_USER + [c for c in fallback_palette if c not in SCENARIO_COLORS_USER]
if len(palette) < n_cases:
    k = int(np.ceil(n_cases / len(palette))); palette = (palette * k)[:n_cases]
case_color = {scen: palette[i] for i, scen in enumerate(case_labels)}

# Baseline & rel-diffs
base_df      = tm_frames[0][[tid_col] + channels_for_plot].set_index(tid_col).sort_index()
base_nonzero = base_df.replace(0, np.nan)

records = []
for scen_label, df in zip(case_labels, tm_frames[1:]):
    if tid_col not in df.columns:
        df = df.assign(**{tid_col: range(len(df))})
    scen_df = df[[tid_col] + channels_for_plot].set_index(tid_col).reindex(base_df.index).sort_index()
    rel = 100.0 * (scen_df - base_df) / base_nonzero
    rel_long = rel.reset_index().melt(id_vars=[tid_col], var_name="Channel", value_name="RelDiff")
    rel_long["Scenario"]    = scen_label
    rel_long["ChannelIdx"]  = rel_long["Channel"].map({c: i for i, c in enumerate(channels_for_plot)})
    rel_long["ChannelLabel"]= rel_long["Channel"].map(label_map)
    records.append(rel_long)

rel_all = pd.concat(records, ignore_index=True)

# --- Figure ------------------------------------------------------------------
fig = go.Figure()
fig.add_hline(y=0, line_width=3, line_color="black")  # zero reference

for scen_ix, scen_label in enumerate(case_labels):
    color = case_color[scen_label]
    fill  = make_transparent(color, BOX_ALPHA)
    df_s  = rel_all[rel_all["Scenario"] == scen_label]

    legend_once = True
    for i, ch in enumerate(channels_for_plot):
        vals = df_s.loc[df_s["Channel"] == ch, "RelDiff"].dropna()
        if vals.empty:
            continue

        x_pos  = x_base[i] + offsets[scen_ix]
        x_left = x_pos - BOX_WIDTH/2
        x_right= x_pos + BOX_WIDTH/2

        # Box (width explicitly set so overlays span edge-to-edge)
        fig.add_trace(go.Box(
            x=np.full(len(vals), x_pos),
            y=vals.values,
            width=BOX_WIDTH,
            name=scen_label,
            boxmean=False,
            line=dict(width=BOX_LINE_WIDTH, color=color),
            fillcolor=fill,
            marker=dict(color=color),
            boxpoints=False,
            showlegend=legend_once,
            legendgroup=f"case::{scen_label}",
            hovertemplate=(
                f"Scenario: {scen_label}<br>"
                f"Channel: {label_map[ch]}<br>"
                "RelDiff: %{y:.2f}%<extra></extra>"
            )
        ))
        legend_once = False

        # Overlays: exactly two lines, edge-to-edge of the box
        if DRAW_MEAN:
            m = vals.mean()
            fig.add_trace(go.Scatter(
                x=[x_left, x_right], y=[m, m], mode="lines",
                line=dict(width=MEAN_LINE_WIDTH, dash=MEAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))
        if DRAW_MEDIAN:
            med = vals.median()
            fig.add_trace(go.Scatter(
                x=[x_left, x_right], y=[med, med], mode="lines",
                line=dict(width=MEDIAN_LINE_WIDTH, dash=MEDIAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))

        # Scatter: to the RIGHT edge of THIS scenario's box for THIS channel
        if SCATTER_OVERLAY:
            df_sc = df_s[df_s["Channel"] == ch]
            if not df_sc.empty:
                x_scatter = x_right + SCATTER_RIGHT_GAP
                # consistent turbine colors (small palette cycle)
                tids = pd.unique(rel_all[tid_col])
                tpal = (px.colors.qualitative.Alphabet
                        + px.colors.qualitative.Light24
                        + px.colors.qualitative.Set3)
                if len(tids) > len(tpal):
                    k = int(np.ceil(len(tids) / len(tpal)))
                    tpal = (tpal * k)[:len(tids)]
                tcol = {tid: tpal[k] for k, tid in enumerate(tids)}

                for tid, df_t in df_sc.groupby(tid_col):
                    fig.add_trace(go.Scatter(
                        x=np.full(len(df_t), x_scatter),
                        y=df_t["RelDiff"].values,
                        mode="markers",
                        marker=dict(size=SCATTER_SIZE, opacity=SCATTER_OPACITY, color=tcol[tid]),
                        showlegend=False,
                        legendgroup=f"case::{scen_label}",
                        hovertemplate=(
                            f"Scenario: {scen_label}<br>"
                            f"Channel: {label_map[ch]}<br>"
                            f"Turbine: {tid}<br>"
                            "RelDiff: %{y:.2f}%<extra></extra>"
                        )
                    ))

if SHOW_LEGEND:
    legend=dict(x=LEGEND_X, y=LEGEND_Y, orientation=LEGEND_ORIENTATION)
else:
    legend = dict(visible=False)
# Layout & categorical ticks
fig.update_layout(
    title="Per-Turbine Relative Differences vs Baseline" if SHOW_TITLE else None,
    xaxis_title="Channel",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    font=dict(size=FONT_SIZE),
    legend=legend,
)

fig.update_xaxes(
    tickmode="array",
    tickvals=x_base,
    ticktext=labels_for_plot,
    tickangle=-45
)

fig.show()


In [16]:
# === Cell 4 : Box plots vs single baseline scalar per channel ===
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

# ----- Figure controls -----
FIG_WIDTH  = 1400
FIG_HEIGHT = 720
FONT_SIZE  = 18

BOX_LINE_WIDTH = 1.6
BOX_ALPHA      = 0.30
BOX_WIDTH      = 0.22          # width of each scenario box
INTRA_CASE_GAP = 0.10          # gap between boxes of different scenarios within a channel
SCATTER_RIGHT_GAP = 0.26       # distance from the RIGHT box edge to the scatter
GROUP_PAD      = 0.40          # extra gap between neighboring channel groups

DRAW_MEAN         = True
DRAW_MEDIAN       = True
MEAN_LINE_WIDTH   = 2.2
MEAN_LINE_DASH    = "solid"
MEDIAN_LINE_WIDTH = 3.0
MEDIAN_LINE_DASH  = "dot"

SCATTER_OVERLAY = False
SCATTER_SIZE    = 6
SCATTER_OPACITY = 0.75

LEGEND_X = 1.02
LEGEND_Y = 1.00
LEGEND_ORIENTATION = "v"
SHOW_LEGEND = True
SHOW_TITLE = False
title="Per-Turbine Relative Differences vs Baseline" if SHOW_TITLE else None,


# ----- Reference choice -----
REF_MODE = "max"       # "min"|"max"|"mean"|"median"|"percentile"
REF_PERCENTILE = 90.0     # used only if REF_MODE == "percentile"

# ----- Utilities / colors -----
try:
    SCENARIO_COLORS_USER
except NameError:
    SCENARIO_COLORS_USER = ["#7f7f7f", "#2ca02c", "#66c2a5", "#1f77b4", "#d62728"]

def make_transparent(color_str, alpha=0.3):
    s = color_str.strip()
    if s.startswith("rgba("):
        r, g, b, *_ = [p.strip() for p in s[5:-1].split(",")]
        return f"rgba({r},{g},{b},{alpha})"
    if s.startswith("rgb("):
        r, g, b = [p.strip() for p in s[4:-1].split(",")]
        return f"rgba({r},{g},{b},{alpha})"
    hex_color = s.lstrip("#")
    r = int(hex_color[0:2], 16); g = int(hex_color[2:4], 16); b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

# ----- Data from Cell 1 -----
TURBINE_METRICS_NAME = "turbine_metrics.parquet"
tm_files  = [str(Path(d) / TURBINE_METRICS_NAME) for d in scenario_dirs]
tm_frames = [pd.read_parquet(p) for p in tm_files]

def detect_tid_col(df):
    for c in df.columns:
        if c.lower() in ("id", "turbine", "turbine_id", "tid"):
            return c
    return None

tid_col = detect_tid_col(tm_frames[0]) or "turbine_id"
if tid_col not in tm_frames[0].columns:
    tm_frames[0] = tm_frames[0].assign(**{tid_col: range(len(tm_frames[0]))})

channels_for_plot = [c for c in channels_ordered if c in tm_frames[0].columns]
if not channels_for_plot:
    raise ValueError("None of the selected channels exist in turbine_metrics.parquet.")
labels_for_plot = [display_name.get(c, c) for c in channels_for_plot]
label_map = dict(zip(channels_for_plot, labels_for_plot))

# ----- Baseline scalar per channel -----
base_df = tm_frames[0][[tid_col] + channels_for_plot].set_index(tid_col).sort_index()
if   REF_MODE == "min":        ref_series = base_df.min(   axis=0)
elif REF_MODE == "max":        ref_series = base_df.max(   axis=0)
elif REF_MODE == "mean":       ref_series = base_df.mean(  axis=0)
elif REF_MODE == "median":     ref_series = base_df.median(axis=0)
elif REF_MODE == "percentile": ref_series = base_df.quantile(REF_PERCENTILE/100.0, axis=0)
else: raise ValueError("REF_MODE must be: min|max|mean|median|percentile")
ref_series = ref_series.replace(0, np.nan)

# ----- Relative diffs vs scalar -----
case_labels = scenario_labels[1:]
n_cases = len(case_labels)

records = []
for scen_label, df in zip(case_labels, tm_frames[1:]):
    if tid_col not in df.columns:
        df = df.assign(**{tid_col: range(len(df))})
    scen_df = df[[tid_col] + channels_for_plot].set_index(tid_col).sort_index()
    ref_df = pd.DataFrame(np.tile(ref_series.values, (len(scen_df), 1)),
                          index=scen_df.index, columns=channels_for_plot)
    rel = 100.0 * (scen_df - ref_df) / ref_df
    rel_long = rel.reset_index().melt(id_vars=[tid_col], var_name="Channel", value_name="RelDiff")
    rel_long["Scenario"]     = scen_label
    rel_long["ChannelIdx"]   = rel_long["Channel"].map({c: i for i, c in enumerate(channels_for_plot)})
    rel_long["ChannelLabel"] = rel_long["Channel"].map(label_map)
    records.append(rel_long)
rel_all = pd.concat(records, ignore_index=True)

# ----- Colors per scenario -----
fallback = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Dark24
palette = SCENARIO_COLORS_USER + [c for c in fallback if c not in SCENARIO_COLORS_USER]
if len(palette) < n_cases:
    k = int(np.ceil(n_cases / len(palette))); palette = (palette * k)[:n_cases]
case_color = {scen: palette[i] for i, scen in enumerate(case_labels)}

# ======= NEW: compute spacing to avoid overlap when many cases =======
# How wide is one channel group (all boxes + right scatter)?
group_span = n_cases*BOX_WIDTH + (n_cases-1)*INTRA_CASE_GAP + SCATTER_RIGHT_GAP
# Base x centers for channels, padded so groups don't touch
x_base = np.arange(len(channels_for_plot)) * (group_span + GROUP_PAD)

# Per-scenario offsets around each channel center, spaced by (BOX_WIDTH + INTRA_CASE_GAP)
if n_cases > 1:
    centers = np.arange(n_cases) - (n_cases-1)/2.0
    offsets = centers * (BOX_WIDTH + INTRA_CASE_GAP)
else:
    offsets = np.array([0.0])

# ----- Plot -----
fig = go.Figure()
fig.add_hline(y=0, line_width=3, line_color="black")

for scen_ix, scen_label in enumerate(case_labels):
    color = case_color[scen_label]
    fill  = make_transparent(color, BOX_ALPHA)
    df_s  = rel_all[rel_all["Scenario"] == scen_label]

    legend_once = True
    for i, ch in enumerate(channels_for_plot):
        vals = df_s.loc[df_s["Channel"] == ch, "RelDiff"].dropna()
        if vals.empty:
            continue

        x_pos   = x_base[i] + offsets[scen_ix]
        x_left  = x_pos - BOX_WIDTH/2
        x_right = x_pos + BOX_WIDTH/2

        # Box (Plotly median only; mean we add)
        fig.add_trace(go.Box(
            x=np.full(len(vals), x_pos),
            y=vals.values,
            width=BOX_WIDTH,
            name=scen_label,
            boxmean=False,
            line=dict(width=BOX_LINE_WIDTH, color=color),
            fillcolor=fill,
            marker=dict(color=color),
            boxpoints=False,
            showlegend=legend_once,
            legendgroup=f"case::{scen_label}",
            hovertemplate=(f"Scenario: {scen_label}<br>"
                           f"Channel: {label_map[ch]}<br>"
                           "RelDiff: %{y:.2f}%<extra></extra>")
        ))
        legend_once = False

        # Overlays
        if DRAW_MEAN:
            m = vals.mean()
            fig.add_trace(go.Scatter(
                x=[x_left, x_right], y=[m, m], mode="lines",
                line=dict(width=MEAN_LINE_WIDTH, dash=MEAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))
        if DRAW_MEDIAN:
            med = vals.median()
            fig.add_trace(go.Scatter(
                x=[x_left, x_right], y=[med, med], mode="lines",
                line=dict(width=MEDIAN_LINE_WIDTH, dash=MEDIAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))

        # Scatter to the right of THIS scenario box
        if SCATTER_OVERLAY:
            df_sc = df_s[df_s["Channel"] == ch]
            if not df_sc.empty:
                x_scatter = x_right + SCATTER_RIGHT_GAP
                tids = pd.unique(rel_all[tid_col])
                tpal = (px.colors.qualitative.Alphabet
                        + px.colors.qualitative.Light24
                        + px.colors.qualitative.Set3)
                if len(tids) > len(tpal):
                    k = int(np.ceil(len(tids) / len(tpal))); tpal = (tpal * k)[:len(tids)]
                tcol = {tid: tpal[k] for k, tid in enumerate(tids)}
                for tid, df_t in df_sc.groupby(tid_col):
                    fig.add_trace(go.Scatter(
                        x=np.full(len(df_t), x_scatter),
                        y=df_t["RelDiff"].values,
                        mode="markers",
                        marker=dict(size=SCATTER_SIZE, opacity=SCATTER_OPACITY, color=tcol[tid]),
                        showlegend=False,
                        legendgroup=f"case::{scen_label}",
                        hovertemplate=(f"Scenario: {scen_label}<br>"
                                       f"Channel: {label_map[ch]}<br>"
                                       f"Turbine: {tid}<br>"
                                       "RelDiff: %{y:.2f}%<extra></extra>")
                    ))

if SHOW_LEGEND:
    legend=dict(x=LEGEND_X, y=LEGEND_Y, orientation=LEGEND_ORIENTATION)
else:
    legend = dict(visible=False)

# Layout & ticks
fig.update_layout(
    title=f"Per-Turbine Relative Differences vs fatigue budget "
          f"{REF_MODE.upper() if REF_MODE!='percentile' else f'P{REF_PERCENTILE:.0f}'}" if SHOW_TITLE else None,
    xaxis_title="Metric",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    font=dict(size=FONT_SIZE),
    legend=legend,
)

fig.update_xaxes(
    tickmode="array",
    tickvals=x_base,              # centers of channel groups
    ticktext=labels_for_plot,
    tickangle=-45
)

fig.show()


In [17]:
# === Cell 5 — Weighted combo metrics (per-scenario) from channel-level stats ===
# Pipeline:
#   1) For each channel: compute baseline scalar across baseline turbines (REF_MODE).
#   2) For each scenario: per-turbine rel.diff to that scalar; aggregate across turbines (AGG_MODE) -> R[channel].
#   3) For each combo: weighted sum over selected channels -> one metric per (scenario, combo).
#   4) Scatter plot (x=combo names, y=metric %), one trace per scenario, with y=0 black line.

import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------------------------

# Define channel combinations (each inner list is one combo)
COMBO_CHANNELS = [
    ["RA_fbrm", "RA_ebrm","RA_blade_torsion"],      # example: bearing family
    ["RA_tbfa", "RA_tbss"],      # example: tower/bending mix
    ["RA_TTM"],
    ["RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed"],  
    ["RA_rsb_l10"],
     ["RA_gsb_l10"],
    [ "RA_BRM", "RA_blade_torsion","RA_TBM", "RA_TTM", "RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed"],   
    # ["Revenue"]
]
    # "RA_ADC", "RA_BRM", "RA_ebrm", "RA_fbrm", "RA_blade_torsion", "RA_tbfa",
    # "RA_tbss", "RA_TBM", "RA_TB_torsion", "RA_ttfa", "RA_TTM", "RA_gsb_l10",
    # "RA_rsb_l10", "RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed", "Energy",
    # "Revenue", "YawTravel"
# Weights for each combo (must be same shape as COMBO_CHANNELS)
COMBO_WEIGHTS = [
        [0.5, 0.3, 0.2],   # weights for combo 
        [0.5, 0.5],  
        [1],   
        [0.3,0.7],
        [1],
        [1],
        [0.2,0.1,0.2,0.2,0.15,0.15],   
        # [1]
    ]

# Display names for the combos (same length as COMBO_CHANNELS)
COMBO_NAMES = [
    "Blades",
    "Tower",
    "Tower top/Nacelle",
    "Shaft",
    "Rotor side bearing",
    "Generator side bearing",
    "Turbine",
    # "Revenue"
]

# Normalize weights per combo to sum to 1?
NORMALIZE_WEIGHTS = True

# Order channels with these always at the end (match Cell 1 convention)
tail_priority = ["PitchADC", "YawTravel", "Energy", "Revenue"]

# ----- Baseline scalar choice per channel  (how to combine turbines in baseline)
# Options: "min", "max", "mean", "median", "percentile"
REF_MODE = "max"
REF_PERCENTILE = 95.0   # used only if REF_MODE == "percentile"

# ----- Aggregate across turbines to get one value per channel & scenario
# Options: "min", "max", "mean", "median", "percentile"
AGG_MODE = "percentile"
AGG_PERCENTILE = 90.0   # e.g. 50 = median; only used if AGG_MODE == "percentile"

# ----- Figure controls (similar to other cells)
FIG_WIDTH  = 1400
FIG_HEIGHT = 600
FONT_SIZE  = 18
MARKER_SIZE = 11
MARKER_SYMBOL = "circle"
MARKER_OPACITY = 0.9
LEGEND_X = 1.02
LEGEND_Y = 1.0
LEGEND_ORIENTATION = "v"
SHOW_LEGEND = True
SHOW_TITLE = True



# Optional: pin first 5 scenario colors (hex or rgb/rgba). Falls back to Plotly palettes after that.
try:
    SCENARIO_COLORS_USER
except NameError:
    SCENARIO_COLORS_USER = ["#7f7f7f", "#2ca02c", "#66c2a5", "#1f77b4", "#d62728"]

SCENARIO_MARKERS_USER = ["circle", "square", "diamond", "cross", "triangle-up"]

# ------------------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------------------

def ordered_channels(chs, tail=tail_priority):
    """Return channels ordered with any in `tail` moved to the end."""
    head = [c for c in chs if c not in tail]
    tail_kept = [c for c in tail if c in chs]
    return head + tail_kept

def detect_tid_col(df):
    for c in df.columns:
        if c.lower() in ("id", "turbine", "turbine_id", "tid"):
            return c
    return None

def agg_series(s: pd.Series, mode: str, p: float | None = None):
    s = s.dropna()
    if s.empty:
        return np.nan
    m = mode.lower()
    if m == "min":       return s.min()
    if m == "max":       return s.max()
    if m == "mean":      return s.mean()
    if m == "median":    return s.median()
    if m == "percentile":
        if p is None:
            raise ValueError("AGG_MODE 'percentile' needs AGG_PERCENTILE")
        return np.percentile(s, p)
    raise ValueError("Unknown aggregation mode")

def ref_reduce(df: pd.DataFrame, mode: str, p: float | None = None) -> pd.Series:
    """Reduce baseline turbines across rows -> one scalar per column (channel)."""
    m = mode.lower()
    if m == "min":        return df.min(axis=0)
    if m == "max":        return df.max(axis=0)
    if m == "mean":       return df.mean(axis=0)
    if m == "median":     return df.median(axis=0)
    if m == "percentile":
        if p is None:
            raise ValueError("REF_MODE 'percentile' needs REF_PERCENTILE")
        return df.quantile(p/100.0, axis=0)
    raise ValueError("Unknown REF_MODE")

# ------------------------------------------------------------------------------
# LOAD DATA (stands alone; only uses Cell 1 globals)
# ------------------------------------------------------------------------------

TURBINE_METRICS_NAME = "turbine_metrics.parquet"
tm_files  = [str(Path(d) / TURBINE_METRICS_NAME) for d in scenario_dirs]
tm_frames = [pd.read_parquet(p) for p in tm_files]

tid_col = detect_tid_col(tm_frames[0]) or "turbine_id"
if tid_col not in tm_frames[0].columns:
    tm_frames[0] = tm_frames[0].assign(**{tid_col: range(len(tm_frames[0]))})

# Figure out which channels from combos actually exist
all_needed_channels = sorted(set(ch for combo in COMBO_CHANNELS for ch in combo))
present_channels = [c for c in all_needed_channels if c in tm_frames[0].columns]
missing_channels = sorted(set(all_needed_channels) - set(present_channels))
if missing_channels:
    print("Missing channels (not found in turbine_metrics.parquet baseline):", missing_channels)

# Limit to present combo channels
COMBO_CHANNELS_PRESENT = [[c for c in combo if c in present_channels] for combo in COMBO_CHANNELS]

# Build list of unique channels we must compute for
channels_for_calc = ordered_channels(present_channels)

# ------------------------------------------------------------------------------
# 1) Build baseline scalar per channel
# ------------------------------------------------------------------------------

base_df = tm_frames[0][[tid_col] + channels_for_calc].set_index(tid_col).sort_index()
ref_series = ref_reduce(base_df, REF_MODE, REF_PERCENTILE if REF_MODE == "percentile" else None)
ref_series = ref_series.replace(0, np.nan)  # avoid division by zero later

# ------------------------------------------------------------------------------
# 2) For each scenario: per-turbine rel.diff to scalar, then aggregate per channel
# ------------------------------------------------------------------------------

case_labels = scenario_labels[1:]
per_channel_metric = {}  # dict: scenario -> pd.Series indexed by channel
for scen_label, df in zip(case_labels, tm_frames[1:]):
    if tid_col not in df.columns:
        df = df.assign(**{tid_col: range(len(df))})
    scen_df = df[[tid_col] + channels_for_calc].set_index(tid_col).sort_index()

    # Compute per-turbine rel.diff to scalar: 100 * (scen - ref)/ref
    ref_df = pd.DataFrame(np.tile(ref_series.values, (len(scen_df), 1)),
                          index=scen_df.index, columns=channels_for_calc)
    reld = 100.0 * (scen_df - ref_df) / ref_df

    # Aggregate across turbines per channel
    agg_vals = {ch: agg_series(reld[ch], AGG_MODE,
                               AGG_PERCENTILE if AGG_MODE == "percentile" else None)
                for ch in channels_for_calc}
    per_channel_metric[scen_label] = pd.Series(agg_vals)

# ------------------------------------------------------------------------------
# 3) Weighted combos -> one metric per (scenario, combo)
# ------------------------------------------------------------------------------

# Normalize weights if requested, handle missing channels per combo gracefully
combo_names = []
combo_metrics = {sc: [] for sc in case_labels}
combo_component_tables = {sc: [] for sc in case_labels}  # for hover text

for idx, (channels, weights, name) in enumerate(zip(COMBO_CHANNELS_PRESENT, COMBO_WEIGHTS, COMBO_NAMES)):
    if len(channels) == 0:
        print(f"⚠️ Combo '{name}' has no valid channels present; results will be NaN.")
        # append NaNs for each scenario
        for sc in case_labels:
            combo_metrics[sc].append(np.nan)
            combo_component_tables[sc].append([])
        combo_names.append(name)
        continue

    # Trim weights to the number of present channels
    if len(weights) < len(channels):
        print(f"⚠️ Combo '{name}' has fewer weights than channels; padding with zeros.")
        weights = weights + [0.0] * (len(channels) - len(weights))
    elif len(weights) > len(channels):
        print(f"⚠️ Combo '{name}' has more weights than channels; extra weights ignored.")
        weights = weights[:len(channels)]

    w = np.array(weights, dtype=float)
    if NORMALIZE_WEIGHTS and np.isfinite(w).any():
        s = w.sum()
        if s != 0:
            w = w / s

    # For each scenario, form weighted sum of per-channel metrics
    for sc in case_labels:
        series_sc = per_channel_metric[sc]
        vals = np.array([series_sc.get(ch, np.nan) for ch in channels], dtype=float)

        # build a small component table for hover
        comp = [(display_name.get(ch, ch), float(v), float(wi)) for ch, v, wi in zip(channels, vals, w)]
        combo_component_tables[sc].append(comp)

        # weighted sum (ignore NaNs by zeroing their weights)
        mask = np.isfinite(vals) & np.isfinite(w)
        metric = (vals[mask] * w[mask]).sum() if mask.any() else np.nan
        combo_metrics[sc].append(metric)

    combo_names.append(name)

# Table of results (rows=scenarios, cols=combo names)
metrics_df = pd.DataFrame({sc: combo_metrics[sc] for sc in case_labels}, index=combo_names).T
print("\nWeighted combo metrics (% rel. diff vs baseline scalar, per scenario):")
print(metrics_df.to_string())

# ------------------------------------------------------------------------------
# 4) Plot scatter (x = combo name, y = metric), one trace per scenario
# ------------------------------------------------------------------------------

# Colors per scenario
fallback = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Dark24
palette = SCENARIO_COLORS_USER + [c for c in fallback if c not in SCENARIO_COLORS_USER]
if len(palette) < len(case_labels):
    k = int(np.ceil(len(case_labels) / len(palette)))
    palette = (palette * k)[:len(case_labels)]
case_color = {sc: palette[i] for i, sc in enumerate(case_labels)}

# --- Marker symbols: define your preferred 5, then cycle if >5 scenarios

case_marker = {sc: SCENARIO_MARKERS_USER[i % len(SCENARIO_MARKERS_USER)]
               for i, sc in enumerate(case_labels)}

fig = go.Figure()
fig.add_hline(y=0, line_width=3, line_color="black")

for sc in case_labels:
    yvals = metrics_df.loc[sc, combo_names].values.astype(float)

    hover_texts = []
    for comps in combo_component_tables[sc]:
        if not comps:
            hover_texts.append(f"<b>{sc}</b><br>Metric: NaN")
            continue
        lines = [f"<b>{sc}</b>"]
        lines.append("<br><b>Components (channel / value% / weight)</b>")
        for ch_label, v, wi in comps:
            if np.isfinite(v):
                lines.append(f"<br>{ch_label}: {v:.2f}% × {wi:.3f}")
            else:
                lines.append(f"<br>{ch_label}: NaN × {wi:.3f}")
        hover_texts.append("".join(lines))

    fig.add_trace(go.Scatter(
        x=combo_names,
        y=yvals,
        mode="markers",
        marker=dict(
            size=MARKER_SIZE,
            symbol=case_marker[sc],     # << per-scenario symbol
            opacity=MARKER_OPACITY,
            color=case_color[sc]
        ),
        name=sc,
        hoverinfo="text+y",
        hovertext=hover_texts,
        showlegend=True
    ))

if SHOW_LEGEND:
    legend=dict(x=LEGEND_X, y=LEGEND_Y, orientation=LEGEND_ORIENTATION)
else:
    legend = dict(visible=False)
    
fig.update_layout(
    title=(f"Weighted Combination Metrics — Baseline {REF_MODE.upper() if REF_MODE!='percentile' else f'P{REF_PERCENTILE:.0f}'} "
           f"/ Across-Turbine {AGG_MODE.upper() if AGG_MODE!='percentile' else f'P{AGG_PERCENTILE:.0f}'}") if SHOW_TITLE else None,
    xaxis_title="Combined metrics",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    font=dict(size=FONT_SIZE),
    legend=legend,
)

fig.show()







Weighted combo metrics (% rel. diff vs baseline scalar, per scenario):
                      Blades      Tower  Tower top/Nacelle      Shaft  Rotor side bearing  Generator side bearing    Turbine
Threshold 500 eur  -2.965595 -15.587982          -1.709551  -3.688224           -3.549336               -3.228355  -3.361995
Threshold 800 eur  -4.779376 -36.792939          -2.214137  -4.811684           -4.717908               -4.336751  -6.335562
Threshold 1000 eur -5.663571 -46.571928          -2.579471  -5.933938           -5.373494               -5.098670  -8.254498
Threshold 1200 eur -6.683976 -53.799979          -3.145116  -7.602192           -5.964021               -5.889360 -10.188170
Threshold 1400 eur -7.615887 -58.659927          -3.783966  -9.558889           -6.496873               -6.683581 -11.905998
Threshold 1600 eur -9.307440 -62.260518          -4.653392 -12.287123           -6.979412               -7.473103 -13.964521


In [3]:
# === Cell 6 — Per-turbine combo metrics (box plots) + lifetime ratios =========
# Logic:
#   R_channel = scenario_value / baseline_scalar(channel)
#   metric%(channel) = 100 * (R_channel - 1)
#   combo per turbine: weighted sum over channels using R (not percents!)
#   metric%(combo)    = 100 * (R_combo - 1)         [Plot 1]
#   lifetime rules per combo:
#       "identity"            -> lifetime_ratio = R_combo
#       "one_minus_plus_one"  -> lifetime_ratio = 2 - R_combo   [Plot 2]

import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------------------------

# COMBO_CHANNELS = [
#     ["RA_fbrm", "RA_ebrm","RA_blade_torsion"],      # example: bearing family
#     ["RA_tbfa", "RA_tbss", "RA_TB_torsion"],      # example: tower/bending mix
#     ["RA_TTM"],
#     ["RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed"],  
#      ["RA_gsb_l10"],
#      ["RA_rsb_l10"]
#     [ "RA_BRM", "RA_blade_torsion","RA_TBM","RA_TB_torsion", "RA_TTM", "RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed"], 
   
#     # ["Revenue"]

    
#     # add more combos if you like...
# ]
#     # "RA_ADC", "RA_BRM", "RA_ebrm", "RA_fbrm", "RA_blade_torsion", "RA_tbfa",
#     # "RA_tbss", "RA_TBM", "RA_TB_torsion", "RA_ttfa", "RA_TTM", "RA_gsb_l10",
#     # "RA_rsb_l10", "RA_shaft_mx_mb_fixed", "RA_shaft_mz_mb_fixed", "Energy",
#     # "Revenue", "YawTravel"
# # Weights for each combo (must be same shape as COMBO_CHANNELS)
# COMBO_WEIGHTS = [
#         [0.5, 0.3, 0.2],   # weights for combo 
#         [0.4, 0.3, 0.3],  
#         [1],   
#         [0.3,0.7],
#         [1],
#         [1],
#         [0.2,0.1,0.1,0.1,0.2,0.15,0.15],   
#         # [1]
#     ]

# # Display names for the combos (same length as COMBO_CHANNELS)
# COMBO_NAMES = [
#     "Blades",
#     "Tower",
#     "Tower top/Nacelle",
#     "Shaft",
#     "Rotor side bearing",
#     "Generator side bearing",
#     "Turbine",
#     # "Revenue"

# ]

# Normalize weights per combo 
NORMALIZE_WEIGHTS = True

# Lifetime rules per combo (same length/order as COMBO_NAMES)
# Allowed: "identity" | "one_minus_plus_one"
LIFETIME_RULES = [
    "one_minus_plus_one",
    "one_minus_plus_one",
    "one_minus_plus_one",
    "one_minus_plus_one",
    "identity",
    "identity",
    "one_minus_plus_one",
]

# Baseline scalar choice (computed from baseline turbines per channel)
# Options: "min" | "max" | "mean" | "median" | "percentile"
REF_MODE = "max"
REF_PERCENTILE = 95.0   # used only if REF_MODE == "percentile"

# --- Plot styling (applies to both figures) -----------------------------------
FIG_WIDTH   = 1400
FIG_HEIGHT  = 720
FONT_SIZE   = 18

# Box geometry / spacing (prevents overlap when many scenarios)
BOX_WIDTH       = 0.32          # width of each scenario box
INTRA_CASE_GAP  = 0.10          # gap between boxes inside a channel group
SCATTER_RIGHT_GAP = 0.24        # reserved space to the right (no scatter by default)
GROUP_PAD       = 0.40          # extra space between channel groups

BOX_LINE_WIDTH  = 1.6
BOX_ALPHA       = 0.30          # box face alpha

# Overlay short lines inside each box
DRAW_MEAN         = True
DRAW_MEDIAN       = True
MEAN_LINE_WIDTH   = 2.2
MEAN_LINE_DASH    = "solid"
MEDIAN_LINE_WIDTH = 3.0
MEDIAN_LINE_DASH  = "dot"

# Legend position
LEGEND_X = 1.02
LEGEND_Y = 1.00
LEGEND_ORIENTATION = "v"
SHOW_LEGEND = True
SHOW_TITLE = True

# Per-scenario color & marker hooks (reused across cells)
try:
    SCENARIO_COLORS_USER
except NameError:
    SCENARIO_COLORS_USER = ["#7f7f7f", "#2ca02c", "#66c2a5", "#1f77b4", "#d62728"]

try:
    SCENARIO_MARKERS_USER
except NameError:
    SCENARIO_MARKERS_USER = ["circle", "square", "diamond", "cross", "triangle-up"]

# ------------------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------------------

def detect_tid_col(df):
    for c in df.columns:
        if c.lower() in ("id", "turbine", "turbine_id", "tid"):
            return c
    return None

def make_transparent(color_str, alpha=0.3):
    s = color_str.strip()
    if s.startswith("rgba("):
        r, g, b, *_ = [p.strip() for p in s[5:-1].split(",")]
        return f"rgba({r},{g},{b},{alpha})"
    if s.startswith("rgb("):
        r, g, b = [p.strip() for p in s[4:-1].split(",")]
        return f"rgba({r},{g},{b},{alpha})"
    # hex
    hex_color = s.lstrip("#")
    r = int(hex_color[0:2], 16); g = int(hex_color[2:4], 16); b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def ref_reduce(df: pd.DataFrame, mode: str, p: float | None = None) -> pd.Series:
    m = mode.lower()
    if m == "min":        return df.min(axis=0)
    if m == "max":        return df.max(axis=0)
    if m == "mean":       return df.mean(axis=0)
    if m == "median":     return df.median(axis=0)
    if m == "percentile":
        if p is None: raise ValueError("REF_MODE 'percentile' requires REF_PERCENTILE")
        return df.quantile(p/100.0, axis=0)
    raise ValueError("Unknown REF_MODE")

# ------------------------------------------------------------------------------
# LOAD DATA (standalone; only uses Cell 1 globals)
# ------------------------------------------------------------------------------

TURBINE_METRICS_NAME = "turbine_metrics.parquet"
tm_files  = [str(Path(d) / TURBINE_METRICS_NAME) for d in scenario_dirs]
tm_frames = [pd.read_parquet(p) for p in tm_files]

tid_col = detect_tid_col(tm_frames[0]) or "turbine_id"
if tid_col not in tm_frames[0].columns:
    tm_frames[0] = tm_frames[0].assign(**{tid_col: range(len(tm_frames[0]))})

# Validate combos & channels availability
all_needed = sorted(set(ch for combo in COMBO_CHANNELS for ch in combo))
present    = [c for c in all_needed if c in tm_frames[0].columns]
missing    = sorted(set(all_needed) - set(present))
if missing:
    print("⚠️ Missing channels in baseline turbine_metrics:", missing)

# Use only present channels per combo
COMBO_CHANNELS_PRESENT = [[c for c in combo if c in present] for combo in COMBO_CHANNELS]

# Baseline scalar per channel (from baseline turbines)
base_df = tm_frames[0][[tid_col] + present].set_index(tid_col).sort_index()
ref_series = ref_reduce(base_df, REF_MODE, REF_PERCENTILE if REF_MODE == "percentile" else None)
ref_series = ref_series.replace(0, np.nan)  # avoid division by zero

# ------------------------------------------------------------------------------
# Compute per-turbine R_channel and per-turbine combo metrics (R_combo, metric%)
# ------------------------------------------------------------------------------

case_labels = scenario_labels[1:]                     # non-baseline scenarios
n_cases     = len(case_labels)

# Colors / markers per scenario
fallback = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Dark24
palette  = SCENARIO_COLORS_USER + [c for c in fallback if c not in SCENARIO_COLORS_USER]
if len(palette) < n_cases:
    k = int(np.ceil(n_cases / len(palette))); palette = (palette * k)[:n_cases]
case_color  = {sc: palette[i] for i, sc in enumerate(case_labels)}
case_marker = {sc: SCENARIO_MARKERS_USER[i % len(SCENARIO_MARKERS_USER)] for i, sc in enumerate(case_labels)}

# For spacing: allocate a channel-group index per combo (x-axis categories)
x_base   = np.arange(len(COMBO_NAMES))
group_span = n_cases*BOX_WIDTH + (n_cases-1)*INTRA_CASE_GAP + SCATTER_RIGHT_GAP
x_base   = x_base * (group_span + GROUP_PAD)

# Offsets per scenario inside a group
if n_cases > 1:
    centers = np.arange(n_cases) - (n_cases-1)/2.0
    offsets = centers * (BOX_WIDTH + INTRA_CASE_GAP)
else:
    offsets = np.array([0.0])

# Build per-scenario, per-combo, per-turbine values
# We'll store two dicts: metric% and lifetime ratio
per_turbine_metricpct = {sc: {name: [] for name in COMBO_NAMES} for sc in case_labels}
per_turbine_ratio     = {sc: {name: [] for name in COMBO_NAMES} for sc in case_labels}

for scen_label, df_scen in zip(case_labels, tm_frames[1:]):
    if tid_col not in df_scen.columns:
        df_scen = df_scen.assign(**{tid_col: range(len(df_scen))})
    scen_df = df_scen[[tid_col] + present].set_index(tid_col).sort_index()

    # R_channel per turbine: scenario / baseline_scalar
    ref_df = pd.DataFrame(np.tile(ref_series.values, (len(scen_df), 1)),
                          index=scen_df.index, columns=present)
    R_channel = scen_df.divide(ref_df)  # division is safe (ref zeros -> NaN)

    # For each combo, compute per-turbine weighted R_combo and metric%
    for combo_idx, (channels, weights, name, rule) in enumerate(zip(COMBO_CHANNELS_PRESENT, COMBO_WEIGHTS, COMBO_NAMES, LIFETIME_RULES)):
        if len(channels) == 0:
            # keep empty -> will remain empty / NaN in plots
            continue

        # Align weights to present channels
        w = np.array(weights[:len(channels)], dtype=float)
        if len(w) < len(channels):
            w = np.pad(w, (0, len(channels)-len(w)), constant_values=0.0)

        # Mask missing channel columns (all NaN) and renormalize if requested
        cols = [c for c in channels if c in R_channel.columns]
        if not cols:
            continue

        R_sub = R_channel[cols].copy()
        # Per-turbine mask for finite values
        finite_mask = np.isfinite(R_sub.values)

        # Normalize weights per combo (global), but reweight per turbine to ignore NaN channels
        w_base = w[:len(cols)]
        if NORMALIZE_WEIGHTS and np.isfinite(w_base).any():
            sum_w = w_base.sum()
            if sum_w != 0:
                w_base = w_base / sum_w

        # Build per-turbine effective weights (zero out where value is NaN then renormalize)
        W_eff = np.tile(w_base, (len(R_sub), 1))
        W_eff[~finite_mask] = 0.0
        row_sums = W_eff.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0.0] = np.nan  # keep NaN if no valid channels for that turbine
        W_eff = W_eff / row_sums

        # Weighted sum to get R_combo per turbine
        R_vals = np.nansum(W_eff * R_sub.values, axis=1)   # shape: (n_turbines,)
        # metric% (for Plot 1)
        metric_pct = 100.0 * (R_vals - 1.0)

        # Lifetime ratio per rule (uses R_vals directly)
        rule_l = rule.lower()
        if rule_l == "identity":
            ratio = R_vals
        elif rule_l == "one_minus_plus_one":
            ratio = 2.0 - R_vals
        else:
            raise ValueError(f"Unknown LIFETIME_RULE '{rule}' for combo '{name}'")

        per_turbine_metricpct[scen_label][name] = metric_pct
        per_turbine_ratio[scen_label][name]     = ratio

# ------------------------------------------------------------------------------
# Plot 1 — Box plot of per-turbine metric% per combo (grouped by scenario)
# ------------------------------------------------------------------------------

fig1 = go.Figure()
fig1.add_hline(y=0, line_width=3, line_color="black")  # zero reference

for scen_ix, sc in enumerate(case_labels):
    color  = case_color[sc]
    fill   = make_transparent(color, BOX_ALPHA)
    symbol = case_marker[sc]
    legend_once = True

    for i, name in enumerate(COMBO_NAMES):
        vals = np.array(per_turbine_metricpct[sc].get(name, []), dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue

        # x placement
        x_pos   = x_base[i] + offsets[scen_ix]
        x_left  = x_pos - BOX_WIDTH/2
        x_right = x_pos + BOX_WIDTH/2

        # Box (median by Plotly), overlays: mean+median short segments inside the box
        fig1.add_trace(go.Box(
            x=np.full(vals.size, x_pos),
            y=vals,
            width=BOX_WIDTH,
            name=sc,
            boxmean=False,
            line=dict(width=BOX_LINE_WIDTH, color=color),
            fillcolor=fill,
            marker=dict(color=color, symbol=symbol),
            boxpoints=False,
            showlegend=legend_once,
            legendgroup=f"case::{sc}",
            hovertemplate=(f"Scenario: {sc}<br>"
                           f"Metric: {name}<br>"
                           "Value: %{y:.2f}%<extra></extra>")
        ))
        legend_once = False

        if DRAW_MEAN:
            m = np.nanmean(vals)
            fig1.add_trace(go.Scatter(
                x=[x_left, x_right], y=[m, m], mode="lines",
                line=dict(width=MEAN_LINE_WIDTH, dash=MEAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))
        if DRAW_MEDIAN:
            med = np.nanmedian(vals)
            fig1.add_trace(go.Scatter(
                x=[x_left, x_right], y=[med, med], mode="lines",
                line=dict(width=MEDIAN_LINE_WIDTH, dash=MEDIAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))
if SHOW_LEGEND:
    legend=dict(x=LEGEND_X, y=LEGEND_Y, orientation=LEGEND_ORIENTATION)
else:
    legend = dict(visible=False)

fig1.update_layout(
    title=("Per-Turbine Combo Metric (%) — "
           f"Baseline {REF_MODE.upper() if REF_MODE!='percentile' else f'P{REF_PERCENTILE:.0f}'}") if SHOW_TITLE else None,
    xaxis_title="Metric (combination)",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    font=dict(size=FONT_SIZE),
    legend=legend,
)

fig1.update_xaxes(
    tickmode="array",
    tickvals=x_base,
    ticktext=COMBO_NAMES,
    tickangle=-45
)
fig1.show()

# ------------------------------------------------------------------------------
# Plot 2 — Box plot of per-turbine lifetime ratio per combo (grouped by scenario)
# ------------------------------------------------------------------------------

fig2 = go.Figure()
# y=1 reference for ratios
fig2.add_hline(y=1.0, line_width=3, line_color="grey")

for scen_ix, sc in enumerate(case_labels):
    color  = case_color[sc]
    fill   = make_transparent(color, BOX_ALPHA)
    symbol = case_marker[sc]
    legend_once = True

    for i, name in enumerate(COMBO_NAMES):
        vals = np.array(per_turbine_ratio[sc].get(name, []), dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue

        x_pos   = x_base[i] + offsets[scen_ix]
        x_left  = x_pos - BOX_WIDTH/2
        x_right = x_pos + BOX_WIDTH/2

        fig2.add_trace(go.Box(
            x=np.full(vals.size, x_pos),
            y=vals,
            width=BOX_WIDTH,
            name=sc,
            boxmean=False,
            line=dict(width=BOX_LINE_WIDTH, color=color),
            fillcolor=fill,
            marker=dict(color=color, symbol=symbol),
            boxpoints=False,
            showlegend=legend_once,
            legendgroup=f"case::{sc}",
            hovertemplate=(f"Scenario: {sc}<br>"
                           f"Metric: {name}<br>"
                           "Lifetime ratio: %{y:.3f}<extra></extra>")
        ))
        legend_once = False

        if DRAW_MEAN:
            m = np.nanmean(vals)
            fig2.add_trace(go.Scatter(
                x=[x_left, x_right], y=[m, m], mode="lines",
                line=dict(width=MEAN_LINE_WIDTH, dash=MEAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))
        if DRAW_MEDIAN:
            med = np.nanmedian(vals)
            fig2.add_trace(go.Scatter(
                x=[x_left, x_right], y=[med, med], mode="lines",
                line=dict(width=MEDIAN_LINE_WIDTH, dash=MEDIAN_LINE_DASH, color=color),
                showlegend=False, hoverinfo="skip"
            ))

fig2.update_layout(
    title=("Per-Turbine Lifetime Ratio — "
           f"Baseline {REF_MODE.upper() if REF_MODE!='percentile' else f'P{REF_PERCENTILE:.0f}'} "
           "/ Rules per metric")if SHOW_TITLE else None,
    xaxis_title="Metric (combination)",
    yaxis_title="Lifetime ratio (-)" ,
    template="plotly_white",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    font=dict(size=FONT_SIZE),
    legend=legend,
)

fig2.update_xaxes(
    tickmode="array",
    tickvals=x_base,
    ticktext=COMBO_NAMES,
    tickangle=-45
)
fig2.show()


FileNotFoundError: [Errno 2] No such file or directory: 'baselines\\TS_FA_HKN_v4_TIboost\\turbine_metrics.parquet'